<a href="https://colab.research.google.com/github/aiwithpavansama/LLM-Model-Creation-from-Scratch/blob/main/LLm_From_Scratch_PreTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# stage 1 : data preparation and sampling , attention mechanism , LLM architecture , Building an LLM
# LLM is auto regressive model / unsupervised leaning . it predicts the next words and then convert that output as input for next prediction.
#it conitues until it completes full sentence

  # 25 concepts to learn in AI
  https://thecuriousmak.substack.com/p/25-ai-concepts-people-use-but-dont

In [ ]:
#tokanization cncept
# step 1: splitting the sentences into individual tokens
# step 2: given the tokens to ID's
# step 3: Convert this into embiddings

In [ ]:
pip install pypdf

In [ ]:
from pypdf import PdfReader

In [ ]:
# reading the text from the PDF and then getting first 100 characters
reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

print("total number of characters",len(text))
print(text[:99])

total number of characters 21813
1
The Verdict
Edith Wharton
1908
Exported from Wikisource on May 20, 20242
I HAD always thought Jac


In [ ]:
#splitting the words based on special characters and removing the spaces.
import re
result=re.split(r'([,.:;?()\'"]|--|\s)',text)
result = [item for item in result if item.strip()]
print(len(result))
print(result[:99])

4871
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called']


In [ ]:
preprocessed=re.split(r'([,.:;?()\'"]|--|\s)',text)
preprocessed=[item for item in preprocessed if item.strip()]
print(len(preprocessed))
print(result[:30])

4871
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough']


In [ ]:
# step 2: given the tokens to ID's (first converting from words to numbers)
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("total number of unique words",vocab_size)


total number of unique words 1276


In [ ]:
vocab={token:Integer for Integer,token in enumerate(all_words)}

In [ ]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('"', 0)
("'", 1)
('(', 2)
(')', 3)
(',', 4)
('--', 5)
('.', 6)
('0', 7)
('1', 8)
('10', 9)
('14', 10)
('19', 11)
('1908', 12)
('1929', 13)
('20', 14)
('2021', 15)
('20242', 16)
('4', 17)
('5', 18)
(':', 19)
(';', 20)
('?', 21)
('A', 22)
('Abigor', 23)
('About', 24)
('AdamBMorgan', 25)
('Ah', 26)
('Among', 27)
('And', 28)
('Are', 29)
('Arrt', 30)
('As', 31)
('At', 32)
('Attribution-ShareAlike', 33)
('AzaToth', 34)
('Be', 35)
('Begin', 36)
('Bender235', 37)
('Blurpeace', 38)
('Boris23', 39)
('Bromskloss', 40)
('Burlington', 41)
('But', 42)
('By', 43)
('Carlo', 44)
('Chicago', 45)
('Claude', 46)
('Come', 47)
('Commons', 48)
('Creative', 49)
('Croft', 50)


In [ ]:
class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, tokens):
        # Convert text tokens to token IDs
        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
        return encoded_ids

    def decode(self, token_ids):
        # Convert token IDs back to tokens
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        import re
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Instantiate and test the tokenizer with the updated vocabulary
tokenizer = SimpleTokenizer(vocab=vocab)

# Example usage with both known and unknown words
test_tokens = preprocessed[:10] + ["<|endoftext|>", "UnknownWordExample"]
print("Original Tokens:", test_tokens)

token_ids = tokenizer.encode(test_tokens)
print("Encoded IDs:", token_ids)

decoded_text = tokenizer.decode(token_ids)
print("Decoded Text:", decoded_text)

Original Tokens: ['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', '<|endoftext|>', 'UnknownWordExample']
Encoded IDs: [8, 142, 156, 59, 162, 12, 61, 576, 166, 839]
Decoded Text: 1 The Verdict Edith Wharton 1908 Exported from Wikisource on


In [ ]:
# we have introduced 2 tokens here , unknow(unk) and endoftext
# unknow is used if any of the input is not in the vocab
# endoftext is used at the end of the sentence or new source of data

In [ ]:
# special tokens special context tokens
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab={token:integer for integer,token in enumerate(all_tokens)}
len(vocab.items())

1278

In [ ]:
#this for testing purpose where , if any of the strint is not match to vocab
#that we trained then it will mapp to ukknown

# Instantiate tokenizer
tokenizer = SimpleTokenizer(vocab=vocab)

# Encode the unknown word "pavan"
encoded_pavan = tokenizer.encode(["Verdict"])
unk_id = vocab['<|unk|>']

# Print evaluation results
print(f"Encoded ID for 'pavan': {encoded_pavan}")
print(f"ID of '<|unk|>': {unk_id}")
print(f"Is 'pavan' correctly mapped to '<|unk|>'? {encoded_pavan == [unk_id]}")

Encoded ID for 'pavan': [156]
ID of '<|unk|>': 1277
Is 'pavan' correctly mapped to '<|unk|>'? False


#Tokenization Algorithm

# **1) word based tokenization- every word is a token **

problems:

1.   what do you do if user enter word which are not in the vocab
2.   when we do word based token simliar words are not group together , they might get token way a far from each other (ex boy , boys both are almost similiar but id's will get so far away )

# **2) character based tokenization:**


1.   each character treats as token
never run of  out of voca(oov) since english has 256 characters


**problems: **


1.   the meaning associated with words are lost
2.   tokenization is larger than initial text

# 3) **sub word tokenization**
split based on rules


*   rule1: donot split frequenlt used words into smaller sub words
*   rule2: split the rare words into smaller meaningful subwords
   
adv:
1.    it handles outof vacob words very well





# **BPE( byte pair encoding ) is a sub word tokenization algorigthm**





In [ ]:
# byte pair encodning
!pip3 install tiktoken

In [ ]:
import tiktoken
import importlib

In [ ]:
tokenizer =tiktoken.get_encoding("gpt2")

In [ ]:
text=("hello , do you like tea?? <|endoftext|> in the sulit terraces")
Integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})

In [ ]:
print(Integers)

[31373, 837, 466, 345, 588, 8887, 3548, 220, 50256, 287, 262, 33154, 270, 8812, 2114]


In [ ]:
strings=tokenizer.decode(Integers)
print(strings)

hello , do you like tea?? <|endoftext|> in the sulit terraces


*#data sampling*
**#Creating input-target pairs**



In [ ]:
reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

print("total number of characters",len(text))
print(text[:99])

total number of characters 21813
1
The Verdict
Edith Wharton
1908
Exported from Wikisource on May 20, 20242
I HAD always thought Jac


In [ ]:
# applying BPE
enc_txt=tokenizer.encode(text)
print(len(enc_txt))

5857


In [ ]:
enc_sample=enc_txt[:100]
print(enc_sample)

[16, 198, 464, 4643, 11600, 198, 7407, 342, 854, 41328, 198, 1129, 2919, 198, 3109, 9213, 422, 11145, 271, 1668, 319, 1737, 1160, 11, 1160, 27877, 198, 40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 198, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 198, 1326, 284, 3285, 326, 11, 287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710, 198, 14363, 12036, 11, 6405, 257, 5527, 27075, 11, 290, 4920, 2241, 198, 259, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561]


In [ ]:
content_size=5  # context size is nothing but how many size in the input and output

x=enc_sample[:content_size]
y=enc_sample[1:content_size+1]
print(x,y)

[16, 198, 464, 4643, 11600] [198, 464, 4643, 11600, 198]


In [ ]:
for i in range(len(enc_sample)-content_size):
  x=enc_sample[i:i+content_size]
  y=enc_sample[i+1:i+content_size+1]
  print("input sending to LLM is ",x,"----->>>","final output after pridcting next words is",y)

input sending to LLM is  [16, 198, 464, 4643, 11600] ----->>> final output after pridcting next words is [198, 464, 4643, 11600, 198]
input sending to LLM is  [198, 464, 4643, 11600, 198] ----->>> final output after pridcting next words is [464, 4643, 11600, 198, 7407]
input sending to LLM is  [464, 4643, 11600, 198, 7407] ----->>> final output after pridcting next words is [4643, 11600, 198, 7407, 342]
input sending to LLM is  [4643, 11600, 198, 7407, 342] ----->>> final output after pridcting next words is [11600, 198, 7407, 342, 854]
input sending to LLM is  [11600, 198, 7407, 342, 854] ----->>> final output after pridcting next words is [198, 7407, 342, 854, 41328]
input sending to LLM is  [198, 7407, 342, 854, 41328] ----->>> final output after pridcting next words is [7407, 342, 854, 41328, 198]
input sending to LLM is  [7407, 342, 854, 41328, 198] ----->>> final output after pridcting next words is [342, 854, 41328, 198, 1129]
input sending to LLM is  [342, 854, 41328, 198, 1129

In [ ]:
for i in range(len(enc_sample)-content_size):
  x=enc_sample[i:i+content_size]
  y=enc_sample[i+1:i+content_size+1]
  print("input sending to LLM is ",tokenizer.decode(x),"----->>>","final output after pridcting next words is",tokenizer.decode(y))

input sending to LLM is  1
The Verdict ----->>> final output after pridcting next words is 
The Verdict

input sending to LLM is  
The Verdict
 ----->>> final output after pridcting next words is The Verdict
Ed
input sending to LLM is  The Verdict
Ed ----->>> final output after pridcting next words is  Verdict
Edith
input sending to LLM is   Verdict
Edith ----->>> final output after pridcting next words is dict
Edith Wh
input sending to LLM is  dict
Edith Wh ----->>> final output after pridcting next words is 
Edith Wharton
input sending to LLM is  
Edith Wharton ----->>> final output after pridcting next words is Edith Wharton

input sending to LLM is  Edith Wharton
 ----->>> final output after pridcting next words is ith Wharton
19
input sending to LLM is  ith Wharton
19 ----->>> final output after pridcting next words is  Wharton
1908
input sending to LLM is   Wharton
1908 ----->>> final output after pridcting next words is arton
1908

input sending to LLM is  arton
1908
 ----->>> f

#DATA LOADER


#we need to do it in a structural format thats why we using data loaders . load and process the data in a compact manner


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return (torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx]))

In [ ]:
import tiktoken
from torch.utils.data import DataLoader

def create_data_loader(txt, tokenizer, max_length=256, stride=128, batch_size=4,drop_last=True,num_workers=0):
  dataset = TextDataset(txt, tokenizer, max_length, stride)
  data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
  return data_loader



In [46]:
reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

print("total number of characters",len(text))

total number of characters 21813


In [47]:
import torch
print("Pytorch version:", torch.__version__)
data_loader = create_data_loader(text, tokenizer, batch_size=2, max_length=4, stride=2)

data_iter = iter(data_loader)
first_batch = next(data_iter)
print(first_batch)

Pytorch version: 2.11.0+cpu
[tensor([[ 1917,    13,  1675, 24456],
        [   11,   484,   198,  6098]]), tensor([[   13,  1675, 24456,   465],
        [  484,   198,  6098, 11137]])]


/tmp/ipykernel_3738/1117999696.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx]))


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[  198, 45168,   470,  1011],
        [  256, 11912,   286, 35394]]), tensor([[45168,   470,  1011,   881],
        [11912,   286, 35394,   739]])]


/tmp/ipykernel_3738/1117999696.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx]))


In [ ]:
# effects of batch size
# more the batch size , good will be the output but needs larger memory size
# less batch means less memeory but more noisey

# Token Embedding


1.   vector embedding : it will catch the similarity(symantic meaning ) btw the words and place it
2.   we have to training neural network to create vector embedding



# we use word to vector "word2vec" and its 300 dimentional vector .it will represent 300 way of words to catch the meaning of each **words**

In [ ]:
!pip install gensim
import gensim.downloader as api
model = api.load("word2vec-google-news-300")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 52.7 MB/s  0:00:00
[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
word_vectors=model
print(word_vectors["cat"]) #vector for cat

[ 0.0123291   0.20410156 -0.28515625  0.21679688  0.11816406  0.08300781
  0.04980469 -0.00952148  0.22070312 -0.12597656  0.08056641 -0.5859375
 -0.00445557 -0.296875   -0.01312256 -0.08349609  0.05053711  0.15136719
 -0.44921875 -0.0135498   0.21484375 -0.14746094  0.22460938 -0.125
 -0.09716797  0.24902344 -0.2890625   0.36523438  0.41210938 -0.0859375
 -0.07861328 -0.19726562 -0.09082031 -0.14160156 -0.10253906  0.13085938
 -0.00346375  0.07226562  0.04418945  0.34570312  0.07470703 -0.11230469
  0.06738281  0.11230469  0.01977539 -0.12353516  0.20996094 -0.07226562
 -0.02783203  0.05541992 -0.33398438  0.08544922  0.34375     0.13964844
  0.04931641 -0.13476562  0.16308594 -0.37304688  0.39648438  0.10693359
  0.22167969  0.21289062 -0.08984375  0.20703125  0.08935547 -0.08251953
  0.05957031  0.10205078 -0.19238281 -0.09082031  0.4921875   0.03955078
 -0.07080078 -0.0019989  -0.23046875  0.25585938  0.08984375 -0.10644531
  0.00105286 -0.05883789  0.05102539 -0.0291748   0.193359

In [ ]:
word_vectors.most_similar(positive=['king', 'men'], negative=['queen'], topn=10, clip_start=0, clip_end=None, restrict_vocab=None, indexer=None)

[('Men', 0.47231149673461914),
 ('man', 0.4697684943675995),
 ('women', 0.4563102126121521),
 ('males', 0.45049843192100525),
 ('brothers', 0.4475797712802887),
 ('boys', 0.4349743127822876),
 ('mens', 0.4187881052494049),
 ('youths', 0.3863909840583801),
 ('fathers', 0.3796182870864868),
 ('platform_Break.com', 0.3772309422492981)]

In [ ]:
#Perform a standard word analogy like 'king' - 'man' + 'woman'
result = word_vectors.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)
print(result)

[('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581)]


# **Positional encoding/embedding**


1.   Absolute
2.   Relative



In [ ]:
vocab_size=50257
output_dim=256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
position_embedding_layer = torch.nn.Embedding(1000, output_dim)

In [52]:
max_length = 4
data_loader = create_data_loader(text, tokenizer, batch_size=8, max_length=4, stride=4)
data_iter = iter(data_loader)
inputs, tragets = next(data_iter)

/tmp/ipykernel_3738/1117999696.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx]))


In [54]:
print("token ids:\n",inputs)
print("input shaoe:\n",inputs.shape)

token ids:
 tensor([[11856,   510,   616, 20348],
        [35403,   701,   262, 40830],
        [  198,  5796,   576,   326],
        [   13,   198,     1, 10049],
        [ 1169,  4252, 18250,  8812],
        [  470,  1309,   340,   198],
        [ 2045,   329,   649,  1866],
        [ 8737,   526,   198, 43920]])
input shaoe:
 torch.Size([8, 4])


In [57]:
token_embedding=token_embedding_layer(inputs).shape
print("token embedding shape:\n",token_embedding)

token embedding shape:
 torch.Size([8, 4, 256])


In [59]:
context_length=max_length
position_embedding_layer=torch.nn.Embedding(context_length,output_dim )

In [60]:
position_embeddings=position_embedding_layer(torch.arange(max_length))
print("position embedding shape:\n",position_embeddings.shape)

position embedding shape:
 torch.Size([4, 256])
